# Stage 01 — Data Ingestion
Prototype notebook for the data ingestion component.
Run cells top to bottom once, then copy the final code to `src/`.

In [ ]:
import os
os.chdir("../")
print("Working directory:", os.getcwd())

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir: Path
    source_URL: str
    local_data_file: Path
    unzip_dir: Path

In [ ]:
from src.creditrisk.constants import *
from src.creditrisk.utils import read_yaml, create_directories, logger

class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion
        create_directories([config.root_dir])
        return DataIngestionConfig(
            root_dir=config.root_dir,
            source_URL=config.source_URL,
            local_data_file=config.local_data_file,
            unzip_dir=config.unzip_dir
        )

In [ ]:
import os
import shutil
from src.creditrisk.utils import logger

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config = config

    def download_data(self):
        """
        Copies the raw CSV from data/raw/ into artifacts/data_ingestion/.
        (Replace with urllib.request.urlretrieve for a real download URL.)
        """
        dest = str(self.config.local_data_file)
        if not os.path.exists(dest):
            src = os.path.join("data", "raw", "credit_risk.csv")
            if os.path.exists(src):
                shutil.copy(src, dest)
                logger.info(f"Data copied from {src} to {dest}")
            else:
                raise FileNotFoundError(f"Place the CSV at: {src}")
        else:
            logger.info(f"File already exists at: {dest}")

    def extract_zip_file(self):
        """No-op for CSV — just confirms the file is present."""
        csv_path = os.path.join(self.config.unzip_dir, "credit_risk.csv")
        if os.path.exists(csv_path):
            logger.info(f"Data file ready at: {csv_path}")
        else:
            raise FileNotFoundError(f"Expected file not found at: {csv_path}")

In [ ]:
try:
    config = ConfigurationManager()
    data_ingestion_config = config.get_data_ingestion_config()
    data_ingestion = DataIngestion(config=data_ingestion_config)
    data_ingestion.download_data()
    data_ingestion.extract_zip_file()
except Exception as e:
    raise e